In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact_manual

cwd = Path.cwd()
if (cwd / 'wishart_levy.py').exists():
    sys.path.insert(0, str(cwd))
elif (cwd / 'random_dnn' / 'wishart_levy.py').exists():
    sys.path.insert(0, str(cwd / 'random_dnn'))

import wishart_levy as wl


In [ ]:
DEFAULT_PARAMS = dict(
    alpha=1.5,
    gamma=1.0,
    entry_scale=1.0,
    normalization='stable',
    n_rows=64,
    num_matrices=8,
    bins=81,
    s_max=6.0,
    num_points=121,
    imag_eps=1e-2,
    quadrature_order=64,
    tail_start=2.0,
)

DEFAULT_PARAMS


In [ ]:
def plot_singular_value_comparison(
    alpha,
    gamma,
    entry_scale,
    normalization,
    n_rows,
    num_matrices,
    bins,
    s_max,
    num_points,
    imag_eps,
    quadrature_order,
    tail_start,
    seed,
    show_empirical=True,
    show_theory=True,
    show_tail=True,
):
    params = dict(
        alpha=float(alpha),
        gamma=float(gamma),
        entry_scale=float(entry_scale),
        normalization=str(normalization),
        n_rows=int(n_rows),
        num_matrices=int(num_matrices),
        bins=int(bins),
        s_max=float(s_max),
        num_points=int(num_points),
        imag_eps=float(imag_eps),
        quadrature_order=int(quadrature_order),
        tail_start=float(tail_start),
        seed=int(seed),
    )

    empirical = (
        wl.empirical_singular_value_spectrum(
            params['alpha'],
            params['gamma'],
            n_rows=params['n_rows'],
            num_matrices=params['num_matrices'],
            entry_scale=params['entry_scale'],
            normalization=params['normalization'],
            bins=params['bins'],
            seed=params['seed'],
            singular_range=(0.0, params['s_max']),
        )
        if show_empirical else None
    )

    theory = (
        wl.theoretical_singular_value_curve(
            params['alpha'],
            params['gamma'],
            s_max=params['s_max'],
            num_points=params['num_points'],
            entry_scale=params['entry_scale'],
            normalization=params['normalization'],
            imag_eps=params['imag_eps'],
            quadrature_order=params['quadrature_order'],
        )
        if show_theory else None
    )

    tail_grid = np.linspace(max(params['tail_start'], 1e-3), params['s_max'], 400)
    tail_density = wl.asymptotic_singular_density(
        tail_grid,
        params['alpha'],
        params['gamma'],
        entry_scale=params['entry_scale'],
        normalization=params['normalization'],
    )
    tail_survival = wl.asymptotic_singular_survival(
        tail_grid,
        params['alpha'],
        params['gamma'],
        entry_scale=params['entry_scale'],
        normalization=params['normalization'],
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    if show_empirical and empirical is not None:
        widths = np.diff(empirical.sv_bin_edges)
        axes[0].bar(
            empirical.sv_bin_centers,
            empirical.sv_density,
            width=widths,
            color='steelblue',
            alpha=0.35,
            label=f'empirical SVD (N={empirical.n_rows}, M={empirical.n_cols}, n={empirical.num_matrices})',
        )
    if show_theory and theory is not None:
        axes[0].plot(
            theory.singular_values,
            theory.singular_density,
            color='black',
            lw=2,
            label='Belinschi singular-value density',
        )
    if show_tail:
        axes[0].plot(
            tail_grid,
            tail_density,
            color='crimson',
            ls=':',
            lw=2,
            label=rf'tail $\sim {wl.singular_tail_prefactor(params["alpha"], params["gamma"], entry_scale=params["entry_scale"], normalization=params["normalization"]):.4f}\,s^{{-(1+{params["alpha"]:.2f})}}$',
        )

    axes[0].set_xlabel(r'$s$')
    axes[0].set_ylabel(r'$f_{\rm sv}(s)$')
    axes[0].set_xlim(0.0, params['s_max'])
    axes[0].set_title(rf'$\alpha={params["alpha"]}$, $\gamma={params["gamma"]}$ ({params["normalization"]})')
    axes[0].legend()

    if show_empirical and empirical is not None:
        mask = (empirical.sv_bin_centers >= params['tail_start']) & (empirical.sv_density > 0.0)
        axes[1].loglog(
            empirical.sv_bin_centers[mask],
            empirical.sv_density[mask],
            'o',
            color='steelblue',
            ms=4,
            label='empirical density',
        )
    if show_theory and theory is not None:
        mask = (theory.singular_values >= params['tail_start']) & (theory.singular_density > 0.0)
        axes[1].loglog(
            theory.singular_values[mask],
            theory.singular_density[mask],
            color='black',
            lw=2,
            label='Belinschi density',
        )
    if show_tail:
        axes[1].loglog(tail_grid, tail_density, color='crimson', ls=':', lw=2, label='tail asymptotic')
    axes[1].set_xlabel(r'$s$')
    axes[1].set_ylabel(r'$f_{\rm sv}(s)$')
    axes[1].set_title('Log-log tail check')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    tail_pref = wl.singular_tail_prefactor(
        params['alpha'], params['gamma'], entry_scale=params['entry_scale'], normalization=params['normalization'],
    )
    survival_pref = wl.singular_survival_tail_prefactor(
        params['alpha'], params['gamma'], entry_scale=params['entry_scale'], normalization=params['normalization'],
    )
    print(f'singular-value tail: f_sv(s) ~ {tail_pref:.6f} * s^(-(1 + {params["alpha"]:.3f}))')
    print(f'survival tail: P(S > s) ~ {survival_pref:.6f} * s^(-{params["alpha"]:.3f})')
    if show_theory and theory is not None:
        print(f'atom at zero (gamma < 1 only): {theory.atom_at_zero:.6f}')
    if show_empirical and empirical is not None:
        print(f'actual matrix shape: N={empirical.n_rows}, M={empirical.n_cols}, gamma={empirical.gamma:.6f}')
    if show_tail:
        q = 0.99
        q_asym = wl.asymptotic_singular_quantile(
            q, params['alpha'], params['gamma'], entry_scale=params['entry_scale'], normalization=params['normalization'],
        )
        print(f'asymptotic upper singular-value quantile Q({q:.2f}) ~ {float(q_asym):.6f}')


In [ ]:
# plot_singular_value_comparison(**DEFAULT_PARAMS, seed=0)

@interact_manual(
    alpha=widgets.FloatText(value=DEFAULT_PARAMS['alpha']),
    gamma=widgets.FloatText(value=DEFAULT_PARAMS['gamma']),
    entry_scale=widgets.FloatText(value=DEFAULT_PARAMS['entry_scale']),
    normalization=widgets.Dropdown(options=['stable', 'belinschi'], value=DEFAULT_PARAMS['normalization']),
    n_rows=widgets.IntText(value=DEFAULT_PARAMS['n_rows']),
    num_matrices=widgets.IntText(value=DEFAULT_PARAMS['num_matrices']),
    bins=widgets.IntText(value=DEFAULT_PARAMS['bins']),
    s_max=widgets.FloatText(value=DEFAULT_PARAMS['s_max']),
    num_points=widgets.IntText(value=DEFAULT_PARAMS['num_points']),
    imag_eps=widgets.FloatText(value=DEFAULT_PARAMS['imag_eps']),
    quadrature_order=widgets.IntText(value=DEFAULT_PARAMS['quadrature_order']),
    tail_start=widgets.FloatText(value=DEFAULT_PARAMS['tail_start']),
    seed=widgets.IntText(value=0),
    show_empirical=widgets.Checkbox(value=True, description='Empirical SVD'),
    show_theory=widgets.Checkbox(value=True, description='Belinschi theory'),
    show_tail=widgets.Checkbox(value=True, description='Tail asymptotic'),
)
def explore_singular_value_density(
    alpha,
    gamma,
    entry_scale,
    normalization,
    n_rows,
    num_matrices,
    bins,
    s_max,
    num_points,
    imag_eps,
    quadrature_order,
    tail_start,
    seed,
    show_empirical,
    show_theory,
    show_tail,
):
    plot_singular_value_comparison(
        alpha=alpha,
        gamma=gamma,
        entry_scale=entry_scale,
        normalization=normalization,
        n_rows=n_rows,
        num_matrices=num_matrices,
        bins=bins,
        s_max=s_max,
        num_points=num_points,
        imag_eps=imag_eps,
        quadrature_order=quadrature_order,
        tail_start=tail_start,
        seed=seed,
        show_empirical=show_empirical,
        show_theory=show_theory,
        show_tail=show_tail,
    )
